# NLP Sentiment Classification on Customer Reviews

Clean review text, extract TF-IDF features, and compare Multinomial Naive Bayes with Logistic Regression.

> **Important:** The provided dataset contains only 10 rows (5 positive, 4 negative, and 1 neutral). This notebook keeps all three labels, but its hold-out scores are only a demonstration of the pipeline and are not reliable estimates of real-world performance.

## 1. Imports and reproducibility

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support,
)

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from sentiment_classifier import (
    build_models,
    clean_review,
    load_reviews,
    predict_sentiment,
    split_reviews,
)

RANDOM_STATE = 42
DATA_PATH = PROJECT_ROOT / "data" / "sentiment_reviews.csv"

## 2. Load and inspect the dataset

The source file includes a neutral label in addition to the requested positive/negative polarity labels. We keep it rather than silently dropping a supplied example.

In [2]:
reviews = load_reviews(DATA_PATH)

print(f"Rows available: {len(reviews)}")
print("Class counts:")
display(reviews["sentiment"].value_counts().rename_axis("sentiment").to_frame("count"))
display(reviews.head())

Rows available: 10
Class counts:


,count
sentiment,
positive,5
negative,4
neutral,1


,review_id,review_text,sentiment
0,1,"The product quality is exceptional, exceeded a...",positive
1,2,Terrible customer service and the item arrived...,negative
2,3,"Average performance, decent for the price poin...",neutral
3,4,"Absolutely love this! Easy to setup, robust, a...",positive
4,5,Stopped working after two days. Extremely disa...,negative


## 3. Text cleaning and lemmatization

The preprocessing function removes HTML, lowercases text, extracts word tokens, removes English stopwords, and lemmatizes tokens. NLTK corpora are downloaded explicitly if needed; if they cannot be obtained, setup fails with an actionable error instead of silently skipping preprocessing.

In [3]:
reviews["clean_text"] = reviews["review_text"].map(clean_review)
display(reviews[["review_text", "clean_text", "sentiment"]].head())

,review_text,clean_text,sentiment
0,"The product quality is exceptional, exceeded a...",product quality exceptional exceeded expectation,positive
1,Terrible customer service and the item arrived...,terrible customer service item arrived complet...,negative
2,"Average performance, decent for the price poin...",average performance decent price point nothing...,neutral
3,"Absolutely love this! Easy to setup, robust, a...",absolutely love easy setup robust work flawlessly,positive
4,Stopped working after two days. Extremely disa...,stopped working two day extremely disappointed...,negative


## 4. Train/test split

A stratified hold-out is useful for a demonstration, but a label with just one example cannot occur in both train and test data. That singleton label is kept in training while the remaining examples are split stratifiably. The resulting test set cannot measure performance for the singleton class.

In [4]:
(
    X_train,
    X_test,
    y_train,
    y_test,
    class_labels,
    singleton_labels,
) = split_reviews(reviews, random_state=RANDOM_STATE)

print(f"Training rows: {len(X_train)}; test rows: {len(X_test)}")
print(f"Labels kept in training because they have one example: {singleton_labels or 'none'}")
print("Test labels:")
display(y_test.value_counts().rename_axis("sentiment").to_frame("count"))

Training rows: 7; test rows: 3
Labels kept in training because they have one example: ['neutral']
Test labels:


,count
sentiment,
positive,2
negative,1


## 5. TF-IDF models

Each pipeline learns its vocabulary and IDF weights from training reviews only. The custom preprocessing function handles token cleanup before TF-IDF vectorization.

In [5]:
models = build_models(random_state=RANDOM_STATE)
for model in models.values():
    model.fit(X_train, y_train)

print("Both TF-IDF classification pipelines are fitted.")

Both TF-IDF classification pipelines are fitted.


## 6. Evaluation and model comparison

The report includes accuracy and weighted/macro precision, recall, and F1. Metrics are computed on the same tiny hold-out set; they should not be interpreted as dependable benchmarks. Classes absent from the test set cannot be assessed.

In [6]:
results = []
predictions = {}

for model_name, model in models.items():
    y_pred = model.predict(X_test)
    predictions[model_name] = y_pred
    weighted = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0
    )
    macro = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0
    )
    results.append(
        {
            "Model": model_name,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Weighted precision": weighted[0],
            "Weighted recall": weighted[1],
            "Weighted F1": weighted[2],
            "Macro precision": macro[0],
            "Macro recall": macro[1],
            "Macro F1": macro[2],
        }
    )

    print(f"\n{model_name} classification report:")
    print(classification_report(y_test, y_pred, labels=class_labels, zero_division=0))
    matrix = confusion_matrix(y_test, y_pred, labels=class_labels)
    display(
        pd.DataFrame(
            matrix,
            index=[f"actual_{label}" for label in class_labels],
            columns=[f"predicted_{label}" for label in class_labels],
        )
    )

comparison = pd.DataFrame(results).set_index("Model").round(3)
display(comparison)
print("Interpret these figures only as a demonstration: the dataset has 10 rows total.")


Multinomial Naive Bayes classification report:
              precision    recall  f1-score   support

    negative       0.50      1.00      0.67         1
     neutral       0.00      0.00      0.00         0
    positive       1.00      0.50      0.67         2

    accuracy                           0.67         3
   macro avg       0.50      0.50      0.44         3
weighted avg       0.83      0.67      0.67         3



,predicted_negative,predicted_neutral,predicted_positive
actual_negative,1,0,0
actual_neutral,0,0,0
actual_positive,1,0,1



Logistic Regression classification report:
              precision    recall  f1-score   support

    negative       0.00      0.00      0.00         1
     neutral       0.00      0.00      0.00         0
    positive       0.67      1.00      0.80         2

    accuracy                           0.67         3
   macro avg       0.22      0.33      0.27         3
weighted avg       0.44      0.67      0.53         3



,predicted_negative,predicted_neutral,predicted_positive
actual_negative,0,0,1
actual_neutral,0,0,0
actual_positive,0,0,2


,Accuracy,Weighted precision,Weighted recall,Weighted F1,Macro precision,Macro recall,Macro F1
Model,,,,,,,
Multinomial Naive Bayes,0.667,0.833,0.667,0.667,0.750,0.75,0.667
Logistic Regression,0.667,0.444,0.667,0.533,0.333,0.50,0.400


Interpret these figures only as a demonstration: the dataset has 10 rows total.


## 7. Interactive prediction helper

`predict_sentiment()` uses the fitted Naive Bayes pipeline, so it applies the same text preprocessing and TF-IDF transformation used during training.

In [7]:
naive_bayes_model = models["Multinomial Naive Bayes"]

examples = [
    "The product is wonderful and works perfectly.",
    "It arrived broken and I am very disappointed.",
    "The product is average and has no special features.",
]
for example in examples:
    print(f"Review: {example}")
    print(predict_sentiment(example, naive_bayes_model), "\n")

Review: The product is wonderful and works perfectly.
{'sentiment': 'positive', 'probabilities': {'negative': 0.3307598017960238, 'neutral': 0.12727272586835428, 'positive': 0.5419674723356218}} 

Review: It arrived broken and I am very disappointed.
{'sentiment': 'negative', 'probabilities': {'negative': 0.531995700206926, 'neutral': 0.12950569551218935, 'positive': 0.33849860428088485}} 

Review: The product is average and has no special features.
{'sentiment': 'positive', 'probabilities': {'negative': 0.3543490864600173, 'neutral': 0.20391239439633155, 'positive': 0.4417385191436514}} 



## 8. Limitations and next steps

- Ten examples are not enough to learn broad language patterns or estimate generalization.
- The neutral class has one record, so it is only represented in training and has no independent test evaluation.
- Results depend heavily on the particular train/test split; do not claim these scores as performance benchmarks.
- Collect a larger, representative, consistently labeled dataset with many examples per sentiment class.
- After expanding the data, use a fixed held-out test set or stratified cross-validation and review per-class precision, recall, and F1 before deployment.